# step03_suna_bottle_calibration - SUNA Stage 3: bottle-bias correction

The co-located profiles from Stage 2 are geometrically correct but still sit on the **raw SUNA offset** (surface
near -2 to -3 uM instead of ~0). This stage removes that bias by comparing the SUNA to the discrete bottle
nitrate from the rosette and fitting a correction, the standard way a SUNA is brought onto an absolute scale
(Johnson et al., 2018; Zheng et al., 2024).

**The method, and why it is a regression and not a single offset.** A SUNA bias has two parts: a fixed offset
(a baseline error, constant with concentration) and a gain (a slope error, proportional to concentration). So
the honest correction fits both: `SUNA = a + k * bottle`, then inverts it, `nitrate_corrected = (SUNA - a) / k`.
On P45_06 cast 06 the fit was `SUNA = -1.19 + 0.884 * bottle`: an offset of -1.2 uM and a gain of 0.88 (the SUNA
reads about 12% low at high nitrate, consistent with the onboard bromide-temperature compensation being off).
A pure offset (`k` forced to 1) is available via `FIT_SLOPE = False` for comparison.

**How SUNA is matched to a bottle.** Bottle trip times on this cruise are recorded as the cast time, not the
exact firing instant, so matching is done by **depth**, not time: for each bottle (cast, depth) the SUNA value
is the median of the L1-clean co-located frames within `DEPTH_WIN` db of that depth (bottle depth in m is taken
as pressure in db, exact to about 0.5% on the shelf). Using the median over a depth window, and both cast legs,
makes the match robust to the response-time lag seen in Stage 2.

**What is usable, from your pairing audit.** Only casts that have BOTH SUNA and bottles can calibrate: on
P45_06 that is CTD_06 (P3), CTD_08 (J2) and CTD_03 (P1). CTD_03 is flagged (59 min pairing gap) and is included
or excluded with `INCLUDE_FLAGGED`. The deep, high-nitrate points come only from CTD_06, so until CTD_08 and
CTD_03 `.cnv` files are added the **slope rests on a single cast**; the notebook states this in its output.

**Calibration file note (stated explicitly, as requested).** The onboard nitrate was computed with
`SNA2503E.cal`, which is not currently in hand. This stage does NOT re-derive nitrate from the spectra, so it
does not use the cal at all: it corrects the onboard nitrate empirically against the bottles. The `SNA2503C.cal`
in the metadata folder is used only for the wavelength grid in the QC stages. When the `SNA2503E.cal` is
retrieved from the instrument, a physics-based temperature and salinity re-derivation (Sakamoto TCSS) can be
inserted before this step; the bottle correction here already absorbs the mean temperature and salinity bias, so
the corrected product is usable in the meantime.

Sources: Johnson et al. (2018), *JTECH* (BGC-Argo nitrate bias correction against reference samples); Zheng et
al. (2024), *L&O: Methods* (bias-corrected high-resolution SUNA profiles from CTD casts); Sakamoto, Johnson and
Coletti (2009) (temperature and salinity dependence, for the deferred re-derivation).

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib

INTERACTIVE_ZOOM = True
try:
    get_ipython().run_line_magic("matplotlib", "widget" if INTERACTIVE_ZOOM else "inline")
except Exception:
    INTERACTIVE_ZOOM = False
    try: get_ipython().run_line_magic("matplotlib", "inline")
    except Exception: matplotlib.use("Agg")
import matplotlib.pyplot as plt
if INTERACTIVE_ZOOM:
    try: plt.ioff()
    except Exception: pass
try: from IPython.display import display, Image
except Exception:
    def display(o): print(o)
    Image=None

# ------------------------------------------------------------------ resolve repo root
try:
    _anchor = Path(__file__).resolve()
except NameError:
    _anchor = Path.cwd()
for _p in (_anchor, *_anchor.parents):
    if (_p / "project_paths.py").exists():
        sys.path.insert(0, str(_p)); break
from project_paths import project_root

CRUISE_ID  = "P45_06"
CTD_ROOT   = project_root()
CRUISE_DIR = CTD_ROOT / "cruises" / CRUISE_ID
L2_FILE    = CRUISE_DIR / "L2" / "SUNA" / f"{CRUISE_ID}__suna_L2_colocated.csv"   # from step02_suna_colocation
BOTTLE_NO3 = CRUISE_DIR / "metadata" / "bottle_nitrate" / f"{CRUISE_ID}_bottle_nitrate.csv"
OUT_DIR    = CRUISE_DIR / "_audit" / "suna"; OUT_DIR.mkdir(parents=True, exist_ok=True)
L3_DIR     = CRUISE_DIR / "L3" / "SUNA";     L3_DIR.mkdir(parents=True, exist_ok=True)

DEPTH_WIN       = 2.5     # db window around each bottle depth to take the SUNA median
MIN_SUNA_IN_WIN = 3       # need at least this many clean SUNA frames to use a bottle point
FIT_SLOPE       = True    # True: fit offset+gain; False: offset only (gain=1)
INCLUDE_FLAGGED = False   # include bottles whose 'flag' column is non-empty
print("cruise:", CRUISE_ID, "| L2:", L2_FILE.name, "| bottles:", BOTTLE_NO3.name, "| L3 out:", L3_DIR)

In [ ]:
# ------------------------------------------------------------------ match SUNA to bottles, fit calibration
L2 = pd.read_csv(L2_FILE, parse_dates=["time_utc"])
bot = pd.read_csv(BOTTLE_NO3)
bot.columns = [c.strip() for c in bot.columns]
if "flag" not in bot.columns: bot["flag"] = ""
bot["flag"] = bot["flag"].fillna("")

def suna_at(cast, depth):
    d = L2[(L2["cast"] == cast) & (L2["L1_clean"] == True) & L2["pressure_db"].notna()]
    m = d[(d["pressure_db"] >= depth - DEPTH_WIN) & (d["pressure_db"] <= depth + DEPTH_WIN)]
    return (m["nitrate_uM"].median(), len(m), m["nitrate_uM"].std()) if len(m) else (np.nan, 0, np.nan)

rows = []
for _, r in bot.iterrows():
    flagged = str(r["flag"]).strip() != ""
    if flagged and not INCLUDE_FLAGGED:
        rows.append(dict(cast=r["cast_id"], station=r.get("station",""), depth_m=r["depth_m"],
                         bottle=r["nitrate_uM"], suna=np.nan, n=0, used=False, note="flagged bottle excluded"))
        continue
    sm, n, sd = suna_at(r["cast_id"], float(r["depth_m"]))
    used = n >= MIN_SUNA_IN_WIN
    rows.append(dict(cast=r["cast_id"], station=r.get("station",""), depth_m=r["depth_m"],
                     bottle=r["nitrate_uM"], suna=round(sm,3) if not np.isnan(sm) else np.nan,
                     suna_sd=round(sd,3) if not np.isnan(sd) else np.nan, n=n, used=used,
                     note=("" if used else ("no SUNA for this cast" if n==0 else f"<{MIN_SUNA_IN_WIN} frames"))))
cal = pd.DataFrame(rows)
fit = cal[cal["used"]]
print(f"{len(fit)} usable bottle-SUNA pairs from {fit['cast'].nunique()} cast(s): {sorted(fit['cast'].unique())}\n")
with pd.option_context("display.width",200):
    print(cal[["cast","station","depth_m","bottle","suna","n","used","note"]].to_string(index=False))

OFFSET = GAIN = np.nan; R2 = np.nan
if len(fit) >= 2:
    b = fit["bottle"].to_numpy(); s = fit["suna"].to_numpy()
    if FIT_SLOPE and len(fit) >= 2 and np.ptp(b) > 0:
        GAIN, OFFSET = np.polyfit(b, s, 1)
    else:
        GAIN, OFFSET = 1.0, float(np.mean(s - b))
    pred = OFFSET + GAIN * b; ss_res = np.sum((s-pred)**2); ss_tot = np.sum((s-s.mean())**2)
    R2 = 1 - ss_res/ss_tot if ss_tot > 0 else np.nan
    resid = s - pred
    print(f"\ncalibration (FIT_SLOPE={FIT_SLOPE}):  SUNA = {OFFSET:.4f} + {GAIN:.4f} * bottle")
    print(f"  offset = {OFFSET:.3f} uM | gain = {GAIN:.4f} | R2 = {R2:.4f} | residual std = {resid.std():.3f} uM | n = {len(fit)}")
    print(f"  correction applied to profiles:  nitrate_corr = (SUNA - ({OFFSET:.4f})) / {GAIN:.4f}")
    if fit["cast"].nunique() < 2:
        print("  CAUTION: calibration rests on a single cast; add CTD_08 and CTD_03 .cnv for a multi-cast fit.")
    hi = fit[fit["bottle"] > 5]
    if hi["cast"].nunique() < 2:
        print("  CAUTION: the gain (slope) is constrained by deep/high-nitrate bottles from one cast only.")
else:
    print("\nnot enough usable pairs to fit; add the .cnv files for casts with bottles (CTD_06, CTD_08, CTD_03).")
cal.to_csv(OUT_DIR / f"{CRUISE_ID}__suna_bottle_calibration.csv", index=False, encoding="utf-8-sig")

In [ ]:
# ------------------------------------------------------------------ apply correction -> L3
if not np.isnan(OFFSET):
    L3 = L2.copy()
    L3["nitrate_corr_uM"] = (L3["nitrate_uM"] - OFFSET) / GAIN
    L3.attrs["offset"] = OFFSET; L3.attrs["gain"] = GAIN
    keep = L3[(L3["in_cast"]==True) & (L3["L1_clean"]==True) & L3["pressure_db"].notna()]
    keep.to_csv(L3_DIR / f"{CRUISE_ID}__suna_L3_corrected.csv", index=False, encoding="utf-8-sig")
    print(f"wrote L3 corrected profiles: {int(len(keep))} in-cast clean frames -> {L3_DIR}")
    print(f"  surface check (corrected NO3 at <10 db): median {keep.loc[keep['pressure_db']<10,'nitrate_corr_uM'].median():.2f} uM (expect ~0)")
else:
    L3 = L2.copy(); L3["nitrate_corr_uM"] = np.nan
    print("no calibration; L3 not written")

In [ ]:
# ------------------------------------------------------------------ calibration + corrected-profile figures
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
if not np.isnan(OFFSET):
    # (1) calibration scatter
    fig=Figure(figsize=(5.4,5.2)); FigureCanvasAgg(fig); ax=fig.subplots()
    for cst,gp in fit.groupby("cast"):
        ax.plot(gp["bottle"],gp["suna"],"o",ms=7,label=cst)
    xx=np.linspace(min(0,fit["bottle"].min()),fit["bottle"].max()*1.05,50)
    ax.plot(xx,OFFSET+GAIN*xx,"-",color="#333",lw=1.3,label=f"fit: {OFFSET:.2f}+{GAIN:.3f}x")
    ax.plot(xx,xx,"--",color="#aaa",lw=1,label="1:1")
    ax.set_xlabel("bottle NO3 (uM)"); ax.set_ylabel("SUNA raw NO3 (uM)")
    ax.set_title(f"{CRUISE_ID} SUNA-bottle calibration (R2={R2:.3f}, n={len(fit)})",fontsize=10)
    ax.legend(fontsize=8,frameon=False)
    for sp in ("top","right"): ax.spines[sp].set_visible(False)
    fig.tight_layout(); out=OUT_DIR/f"{CRUISE_ID}__suna_calibration_scatter.png"; fig.savefig(out,dpi=135)
    try: plt.close(fig)
    except Exception: pass
    if Image is not None:
        try: display(Image(filename=str(out)))
        except Exception: pass
    print("  saved",out.name)
    # (2) corrected profiles per calibrated cast, bottles overlaid
    for cid in sorted(fit["cast"].unique()):
        d=L3[(L3["cast"]==cid)&(L3["L1_clean"]==True)&L3["pressure_db"].notna()]
        bd=bot[bot["cast_id"]==cid]
        if d.empty: continue
        fig=Figure(figsize=(5.0,6.2)); FigureCanvasAgg(fig); ax=fig.subplots()
        ax.plot(d["nitrate_uM"],d["pressure_db"],".",ms=2.5,color="#c9c9c9",label="raw SUNA")
        ax.plot(d["nitrate_corr_uM"],d["pressure_db"],".",ms=2.5,color="#2E6FB0",label="corrected")
        ax.plot(bd["nitrate_uM"],bd["depth_m"],"s",ms=8,color="#d62728",label="bottle",zorder=5)
        ax.invert_yaxis(); ax.axvline(0,color="#bbb",lw=0.6)
        ax.set_xlabel("NO3 (uM)"); ax.set_ylabel("pressure (db)")
        ax.set_title(f"{CRUISE_ID} {cid} corrected",fontsize=10); ax.legend(fontsize=8,frameon=False)
        for sp in ("top","right"): ax.spines[sp].set_visible(False)
        fig.tight_layout(); out=OUT_DIR/f"{CRUISE_ID}__suna_corrected__{cid}.png"; fig.savefig(out,dpi=130)
        try: plt.close(fig)
        except Exception: pass
        if Image is not None:
            try: display(Image(filename=str(out)))
            except Exception: pass
        print("  saved",out.name)

## 5. Calibration statistics

In [ ]:
# ------------------------------------------------------------------ refactor: calibration as a function
# so the statistics and the interactive explorer can recompute it for any matching window / options.
def compute_calibration(depth_win=DEPTH_WIN, fit_slope=FIT_SLOPE, include_flagged=INCLUDE_FLAGGED,
                        min_in_win=MIN_SUNA_IN_WIN):
    def _suna_at(cast, depth):
        d = L2[(L2["cast"]==cast) & (L2["L1_clean"]==True) & L2["pressure_db"].notna()]
        m = d[(d["pressure_db"]>=depth-depth_win) & (d["pressure_db"]<=depth+depth_win)]
        return (m["nitrate_uM"].median(), len(m), m["nitrate_uM"].std()) if len(m) else (np.nan, 0, np.nan)
    rows=[]
    for _, r in bot.iterrows():
        flagged = str(r["flag"]).strip() != ""
        if flagged and not include_flagged:
            rows.append(dict(cast=r["cast_id"], station=r.get("station",""), depth_m=r["depth_m"],
                             bottle=r["nitrate_uM"], suna=np.nan, n=0, used=False, note="flagged excluded"))
            continue
        sm,n,sd=_suna_at(r["cast_id"], float(r["depth_m"]))
        used = n >= min_in_win
        rows.append(dict(cast=r["cast_id"], station=r.get("station",""), depth_m=r["depth_m"],
                         bottle=r["nitrate_uM"], suna=sm, suna_sd=sd, n=n, used=used))
    cal=pd.DataFrame(rows); fit=cal[cal["used"]].dropna(subset=["suna"])
    off=gn=r2=np.nan; cov=None
    if len(fit)>=2:
        b=fit["bottle"].to_numpy(); s=fit["suna"].to_numpy()
        if fit_slope and np.ptp(b)>0:
            (gn,off),cov=np.polyfit(b,s,1,cov=True)
        else:
            gn,off=1.0,float(np.mean(s-b))
        pred=off+gn*b; sstot=np.sum((s-s.mean())**2)
        r2=1-np.sum((s-pred)**2)/sstot if sstot>0 else np.nan
    return cal, fit, off, gn, r2, cov

_T975={1:12.71,2:4.303,3:3.182,4:2.776,5:2.571,6:2.447,7:2.365,8:2.306,9:2.262,10:2.228}
def calibration_stats(fit, off, gn, r2, cov):
    b=fit["bottle"].to_numpy(); s=fit["suna"].to_numpy(); n=len(fit); dof=max(n-2,1)
    corr_at=(s-off)/gn                       # corrected SUNA at the bottle points
    rmse_raw=float(np.sqrt(np.mean((s-b)**2)))
    rmse_cor=float(np.sqrt(np.mean((corr_at-b)**2)))
    bias_raw=float(np.mean(s-b)); bias_cor=float(np.mean(corr_at-b))
    # Pearson r + approx p
    r=float(np.corrcoef(b,s)[0,1]) if np.ptp(b)>0 else np.nan
    tcrit=_T975.get(dof,1.96)
    se_gn=se_off=np.nan
    if cov is not None:
        se_gn=float(np.sqrt(cov[0,0])); se_off=float(np.sqrt(cov[1,1]))
    return dict(n=n, offset=off, gain=gn, R2=r2, pearson_r=r,
                se_offset=se_off, se_gain=se_gn, tcrit=tcrit,
                off_lo=off-tcrit*se_off if np.isfinite(se_off) else np.nan,
                off_hi=off+tcrit*se_off if np.isfinite(se_off) else np.nan,
                gain_lo=gn-tcrit*se_gn if np.isfinite(se_gn) else np.nan,
                gain_hi=gn+tcrit*se_gn if np.isfinite(se_gn) else np.nan,
                rmse_raw_uM=rmse_raw, rmse_corrected_uM=rmse_cor,
                bias_raw_uM=bias_raw, bias_corrected_uM=bias_cor)

In [ ]:
# ------------------------------------------------------------------ calibration statistics
_cal,_fit,_off,_gn,_r2,_cov = compute_calibration()
if np.isfinite(_off):
    st = calibration_stats(_fit,_off,_gn,_r2,_cov)
    print(f"SUNA-bottle calibration statistics  (n = {st['n']} pairs, "
          f"{_fit['cast'].nunique()} cast(s))\n")
    print(f"  model        SUNA = offset + gain * bottle,  corrected = (SUNA - offset) / gain")
    print(f"  offset       {st['offset']:+.3f} uM   (95% CI {st['off_lo']:+.2f} to {st['off_hi']:+.2f})")
    print(f"  gain         {st['gain']:.4f}      (95% CI {st['gain_lo']:.3f} to {st['gain_hi']:.3f})")
    print(f"  R2           {st['R2']:.4f}       Pearson r {st['pearson_r']:.4f}")
    print(f"  RMSE vs bottle:  raw SUNA {st['rmse_raw_uM']:.3f} uM  ->  corrected {st['rmse_corrected_uM']:.3f} uM")
    print(f"  mean bias:       raw SUNA {st['bias_raw_uM']:+.3f} uM  ->  corrected {st['bias_corrected_uM']:+.3f} uM")
    print(f"  Zheng (2024) final-QC RMSE reference: 0.34 to 0.78 uM")
    if st['n'] < 10:
        print(f"\n  NOTE: n={st['n']} (< 10). The 95% CIs are wide and the gain leans on the deep")
        print(f"        CTD_06 bottles; treat offset as well constrained, gain as provisional.")
    # per-point residual table
    _fit2=_fit.copy()
    _fit2["corrected"]=(_fit2["suna"]-_off)/_gn
    _fit2["resid_uM"]=_fit2["corrected"]-_fit2["bottle"]
    print("\nper-bottle residuals after correction:")
    with pd.option_context("display.width",200):
        print(_fit2[["cast","depth_m","bottle","suna","corrected","resid_uM","n"]]
              .round(3).to_string(index=False))
    pd.DataFrame([st]).to_csv(OUT_DIR/f"{CRUISE_ID}__suna_calibration_stats.csv",index=False,encoding="utf-8-sig")
    print("\n  saved", f"{CRUISE_ID}__suna_calibration_stats.csv")
else:
    print("no calibration to summarize")

## 6. Interactive calibration explorer

In [ ]:
# ------------------------------------------------------------------ interactive: calibration explorer
# Tune the depth-matching window and fit options and watch the calibration respond. Left: SUNA-bottle
# scatter with the live fit; right: the selected cast's corrected profile with its bottles.
try:
    import ipywidgets as W
    _HAVE_W=True
except Exception:
    _HAVE_W=False; print("ipywidgets not installed; pip install ipywidgets ipympl then Restart + Run All.")

def _explorer_png(cast_sel, depth_win, fit_slope, include_flagged):
    import io
    cal,fit,off,gn,r2,cov = compute_calibration(depth_win,fit_slope,include_flagged)
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    fig=Figure(figsize=(12.5,6.0)); FigureCanvasAgg(fig); ax1,ax2=fig.subplots(1,2)
    # left: calibration scatter
    if len(fit)>=2 and np.isfinite(off):
        for cst,gp in fit.groupby("cast"):
            ax1.plot(gp["bottle"],gp["suna"],"o",ms=7,label=cst)
        xx=np.linspace(min(0,fit["bottle"].min()),fit["bottle"].max()*1.05,50)
        ax1.plot(xx,off+gn*xx,"-",color="#333",lw=1.3,label=f"fit {off:.2f}+{gn:.3f}x")
        ax1.plot(xx,xx,"--",color="#aaa",lw=1,label="1:1")
        st=calibration_stats(fit,off,gn,r2,cov)
        ax1.set_title(f"calibration  R2={r2:.3f}  RMSE {st['rmse_raw_uM']:.2f}->{st['rmse_corrected_uM']:.2f} uM  n={len(fit)}",fontsize=9)
    else:
        ax1.text(0.5,0.5,"< 2 usable pairs",ha="center"); 
    ax1.set_xlabel("bottle NO3 (uM)"); ax1.set_ylabel("SUNA raw NO3 (uM)")
    ax1.legend(fontsize=7,frameon=False)
    for sp in ("top","right"): ax1.spines[sp].set_visible(False)
    # right: corrected profile for the chosen cast
    if np.isfinite(off) and cast_sel!="(none)":
        d=L2[(L2["cast"]==cast_sel)&(L2["L1_clean"]==True)&L2["pressure_db"].notna()].copy()
        d["corr"]=(d["nitrate_uM"]-off)/gn
        bd=bot[bot["cast_id"]==cast_sel]
        ax2.plot(d["nitrate_uM"],d["pressure_db"],".",ms=2.3,color="#c9c9c9",label="raw")
        ax2.plot(d["corr"],d["pressure_db"],".",ms=2.3,color="#2E6FB0",label="corrected")
        ax2.plot(bd["nitrate_uM"],bd["depth_m"],"s",ms=8,color="#d62728",label="bottle",zorder=5)
        ax2.invert_yaxis(); ax2.axvline(0,color="#bbb",lw=0.6)
        ax2.set_title(f"{cast_sel} profile (window +/-{depth_win:.1f} db)",fontsize=9)
    ax2.set_xlabel("NO3 (uM)"); ax2.set_ylabel("pressure (db)"); ax2.legend(fontsize=7,frameon=False)
    for sp in ("top","right"): ax2.spines[sp].set_visible(False)
    fig.suptitle(f"{CRUISE_ID} SUNA-bottle calibration explorer",fontsize=11)
    fig.tight_layout(rect=[0,0,1,0.96])
    buf=io.BytesIO(); fig.savefig(buf,format="png",dpi=118); buf.seek(0)
    try: plt.close(fig)
    except Exception: pass
    return buf.getvalue()

if _HAVE_W:
    _casts=sorted(bot["cast_id"].unique().tolist())
    wcast=W.Dropdown(options=_casts+["(none)"],value=_casts[0] if _casts else "(none)",description="cast")
    wwin =W.FloatSlider(min=0.5,max=5.0,step=0.5,value=DEPTH_WIN,description="depth +/- db",
                        continuous_update=False,layout=W.Layout(width="45%"))
    wslope=W.Checkbox(value=FIT_SLOPE,description="fit gain (slope)")
    wflag =W.Checkbox(value=INCLUDE_FLAGGED,description="include flagged bottles")
    def _r(cast_sel,depth_win,fit_slope,include_flagged):
        display(Image(data=_explorer_png(cast_sel,depth_win,fit_slope,include_flagged)))
    out=W.interactive_output(_r,{"cast_sel":wcast,"depth_win":wwin,"fit_slope":wslope,"include_flagged":wflag})
    display(W.VBox([W.HBox([wcast,wwin]),W.HBox([wslope,wflag]),out]))